### Demo

In [1]:
import os
import numpy as np
import pandas as pd
import random
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchcam.methods import GradCAM

import sklearn
import pickle
import matplotlib.pyplot as plt
from PIL import Image

In [2]:
device = "cuda:1" if torch.cuda.is_available() else "cpu"
# device = "cpu"

In [3]:
random.seed(1234)
torch.manual_seed(1234)

#### Model Flag

In [4]:
# MODEL_FLAG = 'resnet'
# MODEL_FLAG = 'EfficientNet_V2'
# MODEL_FLAG = 'MobileNet_V3'
# MODEL_FLAG = 'SwinTransformer'
# MODEL_FLAG = 'ViT'
MODEL_FLAG = 'ConvNeXt'
# MODEL_FLAG = 'maxvit'

IS_FUSION = True
IS_ATTENTION = True

In [5]:
pretrain = True

#### Model Builder

In [ ]:
if MODEL_FLAG == 'resnet':
    from torchvision.models import resnet18, ResNet18_Weights

    if pretrain:
        model = resnet18(weights="IMAGENET1K_V1")
    else:
        model = resnet18(weights=None)

elif MODEL_FLAG == 'SwinTransformer':
    from torchvision.models import swin_v2_t, Swin_V2_T_Weights

    if pretrain:
        model = swin_v2_t(weights="IMAGENET1K_V1")
    else:
        model = swin_v2_t(weights=None)

elif MODEL_FLAG == 'EfficientNet_V2':
    from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

    if pretrain:
        model = efficientnet_v2_s(weights="IMAGENET1K_V1")
    else:
        model = efficientnet_v2_s(weights=None)

elif MODEL_FLAG == 'MobileNet_V3':
    from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

    if pretrain:
        model = mobilenet_v3_small(weights="IMAGENET1K_V1")
    else:
        model = mobilenet_v3_small(weights=None)

elif MODEL_FLAG == 'ViT':
    from torchvision.models import vit_b_16, ViT_B_16_Weights

    if pretrain:
        model = vit_b_16(weights="IMAGENET1K_V1")
    else:
        model = vit_b_16(weights=None)

elif MODEL_FLAG == 'ConvNeXt':
    from torchvision.models import convnext_small, ConvNeXt_Small_Weights

    if pretrain:
        model = convnext_small(weights="IMAGENET1K_V1")
    else:
        model = convnext_small(weights=None)
        
elif MODEL_FLAG == 'maxvit':
    from torchvision.models import maxvit_t, MaxVit_T_Weights

    if pretrain:
        model = maxvit_t(weights="IMAGENET1K_V1")
    else:
        model = maxvit_t(weights=None)

In [6]:
def cross_attention(Q, K, V, mask=None):
    # Compute the dot products between Q and K, then scale
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / torch.sqrt(torch.tensor(d_k, dtype=torch.float32))

    # Apply mask if provided
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))

    # Softmax to normalize scores and get attention weights
    attention_weights = F.softmax(scores, dim=-1)

    # Weighted sum of values
    output = torch.matmul(attention_weights, V)
    return output, attention_weights

In [7]:
#add another fully connected layer to convert output (1,1000) to (1)
class GaResnetModel(nn.Module):
    def __init__(self, pretrain_model):
        super().__init__()
        self.pretrained_model = pretrain_model # delete the last fc layer.
        self.regression = nn.Linear(1000, 256)

    def forward(self, x):
        # input size of x (b, 3, 256, 256) where b is the batch size
        x = self.pretrained_model(x) #output size (b, 1000)
        x = nn.ReLU()(x)
        x = self.regression(x) #output size (1)

        return x

In [8]:
class AttenFusionModel(nn.Module):
    def __init__(self, img_model, embed_size=512, num_classes=1):
        super().__init__()
        self.img_model = img_model
        self.linear1 = nn.Linear(1, embed_size)
        self.linear2 = nn.Linear(1, embed_size)
        self.linear3 = nn.Linear(1, embed_size)
        # self.flatten = nn.Flatten()
        self.MLP = nn.Sequential(
            # nn.Linear(in_channel, num_classes),
            # nn.ReLU(),
            nn.Linear(embed_size*95, embed_size),
            nn.ReLU(),
            nn.Linear(embed_size, num_classes),
        )

    def forward(self, x1, x2):
        x1 = self.img_model(x1)
        # x = torch.cat((o1, x2), 1)
        x1 = x1[:, :, None]
        x2 = x2[:, :, None]

        Q = self.linear1(x2)
        K = self.linear2(x1)
        V = self.linear3(x1)

        out, attention_weights = cross_attention(Q, K, V)
        # print(out.shape)
        # out = self.flatten(out)
        # print(out.shape)
        out = self.MLP(out.flatten(1))
        # out = self.MLP(out)

        return out, attention_weights

In [ ]:
# 512
# 256
embed = 256 # 512, 256
img_model = GaResnetModel(model)
model = AttenFusionModel(img_model, embed_size=embed) # 256+95

In [ ]:
# model

In [ ]:
def adopt_gradcam(model, input1=None, input2=None):
    # Forward pass
    # Choose target layer from ConvNeXt (e.g., the last block before the head)
    target_layer = model.img_model.pretrained_model.features[-1][-1].block[-1]
    # Create GradCAM instance
    cam_extractor = GradCAM(model, target_layer=target_layer)

    out, attn_weights = model(input1, input2)
    class_idx = out.argmax().item()

    # Compute CAM
    activation_map = cam_extractor(class_idx, out)

    # activation_map[0]: shape (1, 7, 7)
    cam = activation_map[0]     # shape: (1, 7, 7)
    cam = F.interpolate(cam.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False)
    cam = cam.squeeze().cpu().numpy()

    # Normalize to [0, 1]
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    # cam_extractor.close()
    return cam

##### Test Model

In [ ]:
# batch, seq, feature
# src1 = torch.rand(8, 3, 256, 256)
src1 = torch.rand(1, 3, 224, 224) # ViT
src2 = torch.rand(1, 95)

In [ ]:
model = model.to(device)
src1 = src1.to(device)
src2 = src2.to(device)
out, attn_weights = model(src1, src2)
print(out.shape)
print(attn_weights.shape)

Test GradCAM functionality.

In [ ]:
# Compute CAM
# activation_map_cam = adopt_gradcam(model, src1, src2)

In [ ]:
# test_img = Image.open("/mnt/storage/fangyijie/HC18/image/792_HC.png").convert("RGB")
# test_img = test_img.resize((224, 224))  # Resize to match model input size
# test_img.size

In [ ]:
# # Visualize
# plt.imshow(test_img)
# plt.imshow(activation_map_cam, cmap='jet', alpha=0.5)
# plt.title("Grad-CAM Heatmap")
# plt.axis('off')
# plt.show()

### Dataset

In [9]:
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler, Subset
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

In [10]:
TR_TRANSFORMS = transforms.Compose([
    # transforms.Resize((256, 256)),
    transforms.Resize((224, 224)),
    transforms.RandomRotation(degrees=(-15, 15)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

VAL_TRANSFORMS = transforms.Compose([
    # transforms.Resize((256, 256)),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

In [11]:
root_dir = '/mnt/storage/fangyijie/radiomics_ga'

df_dir = root_dir + '/ga_data'
img_dir = root_dir + '/img_data'

In [12]:
X_train = pd.read_pickle(os.path.join(df_dir, 'X_train.pkl'))
y_train = pd.read_pickle(os.path.join(df_dir, 'y_train.pkl'))

X_test = pd.read_pickle(os.path.join(df_dir, 'X_test.pkl'))
y_test = pd.read_pickle(os.path.join(df_dir, 'y_test.pkl'))

In [13]:
# ignore column 'image_name'
# convert dataframe to matrix
src_X_array = X_train.iloc[:, :-1].to_numpy()
src_y_array = y_train.to_numpy()

In [14]:
from sklearn.preprocessing import StandardScaler
# Standardizing data
scaler = StandardScaler()
scaler.fit(src_X_array)
# src_X_array = scaler.transform(src_X_array)

StandardScaler()

Datasets

In [15]:
from src.dataset.img_radiomics import ImgRadiomicDataset

In [16]:
val_size = 100

In [17]:
train_data = ImgRadiomicDataset(root_dir=root_dir+'/img_data', 
    df1=X_train, df2=y_train, 
    rad_df=X_train,
    scaler=scaler,
    train=True, 
    transform=TR_TRANSFORMS
)

test_data = ImgRadiomicDataset(root_dir=root_dir+'/img_data', 
    df1=X_test, df2=y_test,
    rad_df=X_test,
    scaler=scaler,
    train=False, 
    transform=VAL_TRANSFORMS
)

val_data = Subset(test_data, range(val_size))

In [ ]:
print(len(train_data))
print(len(val_data))
print(len(test_data))

Data Loader

In [18]:
batchsize = 8

In [19]:
# Define data loaders for training and testing data in this fold
train_dataloader = DataLoader(train_data, batch_size=batchsize, shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=batchsize, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=batchsize, shuffle=True)

In [ ]:
for X, y, z in train_dataloader:
        print("Shape of X: ", X.shape, X.dtype)
        # Output: Shape of X:  torch.Size([1, 1, 543]) torch.float64
        print("Shape of y: ", y.shape, y.dtype)
        print("Shape of z: ", z.shape, z.dtype)
        break

### Training

In [ ]:
import tqdm

In [ ]:
# embed = 512 # 512, 256
# img_model = GaResnetModel(model)
# model = AttenFusionModel(img_model, embed_size=embed) # 256+95

In [ ]:
num_epochs = 60
warm_up_epoch = 30
lr = 0.00001
# lr = 0.0005 # ViT
# lr = 0.0001 # MLP
criterion = nn.MSELoss() # MSE loss

In [ ]:
len(train_dataloader)

In [ ]:
# GA
if MODEL_FLAG == 'cnn':
    log_dir = root_dir + '/logs/ga/1d_cnn'

if MODEL_FLAG == 'transformer':
    log_dir = root_dir + '/logs/ga/1d_transformer'

if MODEL_FLAG == 'mlp':
    log_dir = root_dir + '/logs/ga/mlp'

if MODEL_FLAG == 'attention':
    log_dir = root_dir + '/logs/ga/attention'

if MODEL_FLAG == 'resnet':
    log_dir = root_dir + '/logs/ga/resnet'

if MODEL_FLAG == 'SwinTransformer':
    log_dir = root_dir + '/logs/ga/SwinTransformer'

if MODEL_FLAG == 'EfficientNet_V2':
    log_dir = root_dir + '/logs/ga/efficientnet'

if MODEL_FLAG == 'MobileNet_V3':
    log_dir = root_dir + '/logs/ga/mobilenet'

if MODEL_FLAG == 'ViT':
    log_dir = root_dir + '/logs/ga/vit'

if MODEL_FLAG == 'ConvNeXt':
    log_dir = root_dir + '/logs/ga/ConvNeXt'

if MODEL_FLAG == 'maxvit':
    log_dir = root_dir + '/logs/ga/maxvit'   

log_dir

In [ ]:
ckpt_dir = root_dir + '/ckpt'
ckpt_dir

In [ ]:
# Initialize optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-6)
# optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9) #ViT

In [ ]:
model = model.to(device)

losses_train: list = []
losses_val: list = []
tr_size = len(train_dataloader)
val_size = len(val_dataloader)
best_loss = 100.0

for epoch in range(0, num_epochs):
    print(f"Epoch {epoch + 1}\n----------------------------")
    tr_loss = 0
    model.train()
    for batch, (x1, x2, y) in enumerate(train_dataloader):
        x1, x2, y = x1.to(device), x2.to(device), y.to(device)

        # Compute the prediction error
        if IS_ATTENTION:
            pred, att_wgh = model(x1, x2)
        else:
            pred = model(x1, x2)
        # print(y)
        # print(pred)
        # loss = criterion(pred, y) # MSE
        loss = torch.sqrt(criterion(pred, y)) # RMSE

        # break
        
        # Reset the gradient to 0 after each epoch
        optimizer.zero_grad()

        # Compute the backpropagation (back propagate the loss and add one step to the optimizer)
        loss.backward()
        optimizer.step()

        loss = loss.item()
        tr_loss += loss

    print(f"training loss: {tr_loss/tr_size:>7f}")
    losses_train.append(tr_loss/tr_size)

    # validation
    val_loss = 0
    model.eval()

    with torch.no_grad():
        for x1, x2, y in val_dataloader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            if IS_ATTENTION:
                pred, att_wgh = model(x1, x2)
            else:
                pred = model(x1, x2)
            loss = torch.sqrt(criterion(pred, y)).item() # RMSE
            # loss = criterion(pred, y).item() # MSE
            val_loss += loss
            # break
    losses_val.append(val_loss/val_size)
    print(f"Validation loss: {val_loss/val_size:>8f} \n")

    # save model
    current_time = time.strftime("%Y%m%d-%H%M%S")
    saved_name = f"checkpoint_epoch_{epoch}_{MODEL_FLAG}_pretrain{pretrain}_atten{IS_ATTENTION}_{embed}emb_fusion_{current_time}.pt"
        
    new_loss = val_loss/val_size
    if new_loss < best_loss:
        best_loss = new_loss
        if epoch > warm_up_epoch:
            torch.save(model.state_dict(), os.path.join(ckpt_dir, saved_name))

# save loss locally
if IS_FUSION:
    with open(f'{log_dir}/imgdata_{epoch}_fusion_train_loss_{current_time}.obj', 'wb') as fp:
        pickle.dump(losses_train, fp)

    with open(f'{log_dir}/imgdata_{epoch}_fusion_val_loss_{current_time}.obj', 'wb') as fp:
        pickle.dump(losses_val, fp)
else:
    with open(f'{log_dir}/imgdata_{epoch}_train_loss_{current_time}.obj', 'wb') as fp:
        pickle.dump(losses_train, fp)
    
    with open(f'{log_dir}/imgdata_{epoch}_val_loss_{current_time}.obj', 'wb') as fp:
        pickle.dump(losses_val, fp)

### Plotting RMSE

In [ ]:
import matplotlib.pyplot as plt

Plot the training loss and validation loss

In [ ]:
axis_x = np.arange(0, len(losses_train), 1, dtype=int)
axis_y = losses_train

axis_val_x = np.arange(0, len(losses_val), 1, dtype=int)
axis_val_y = losses_val

In [ ]:
# Plot the chart
plt.plot(axis_x, axis_y, label="train")
plt.plot(axis_val_x, axis_val_y, label="validation")
plt.legend(loc="upper right")
plt.show()  # display

### Load the Best Model

In [20]:
MODEL_FLAG='ConvNeXt'

In [21]:
embed=512

root_dir = '/mnt/storage/fangyijie/radiomics_ga'

df_dir = root_dir + '/ga_data'
img_dir = root_dir + '/img_data'

In [22]:
ckpt_dir = root_dir + '/ckpt'
ckpt_dir

'/mnt/storage/fangyijie/radiomics_ga/ckpt'

In [23]:
# checkpoint_epoch_41_resnet_pretrainTrue_fusion_20250129-141412.pt
# checkpoint_epoch_57_MobileNet_V3_pretrainTrue_attenTrue_512emb_fusion_20250205-104324.pt
# checkpoint_epoch_49_ConvNeXt_pretrainTrue_attenTrue_256emb_fusion_20250623-204524.pt
# checkpoint_epoch_56_ConvNeXt_pretrainTrue_attenTrue_512emb_fusion_20250623-200725.pt
# checkpoint_epoch_41_ConvNeXt_pretrainTrue_attenTrue_512emb_fusion_20250130-142831.pt

best_model = ckpt_dir + '/checkpoint_epoch_56_ConvNeXt_pretrainTrue_attenTrue_512emb_fusion_20250623-200725.pt'
best_model

'/mnt/storage/fangyijie/radiomics_ga/ckpt/checkpoint_epoch_56_ConvNeXt_pretrainTrue_attenTrue_512emb_fusion_20250623-200725.pt'

In [24]:
if MODEL_FLAG == 'resnet':
    from torchvision.models import resnet18, ResNet18_Weights

    if pretrain:
        model = resnet18(weights="IMAGENET1K_V1")
    else:
        model = resnet18(weights=None)

elif MODEL_FLAG == 'SwinTransformer':
    from torchvision.models import swin_v2_t, Swin_V2_T_Weights

    if pretrain:
        model = swin_v2_t(weights="IMAGENET1K_V1")
    else:
        model = swin_v2_t(weights=None)

elif MODEL_FLAG == 'EfficientNet_V2':
    from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights

    if pretrain:
        model = efficientnet_v2_s(weights="IMAGENET1K_V1")
    else:
        model = efficientnet_v2_s(weights=None)

elif MODEL_FLAG == 'MobileNet_V3':
    from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

    if pretrain:
        model = mobilenet_v3_small(weights="IMAGENET1K_V1")
    else:
        model = mobilenet_v3_small(weights=None)

elif MODEL_FLAG == 'ViT':
    from torchvision.models import vit_b_16, ViT_B_16_Weights

    if pretrain:
        model = vit_b_16(weights="IMAGENET1K_V1")
    else:
        model = vit_b_16(weights=None)

elif MODEL_FLAG == 'ConvNeXt':
    from torchvision.models import convnext_small, ConvNeXt_Small_Weights

    if pretrain:
        model = convnext_small(weights="IMAGENET1K_V1")
    else:
        model = convnext_small(weights=None)
        
elif MODEL_FLAG == 'maxvit':
    from torchvision.models import maxvit_t, MaxVit_T_Weights

    if pretrain:
        model = maxvit_t(weights="IMAGENET1K_V1")
    else:
        model = maxvit_t(weights=None)

In [25]:
img_model = GaResnetModel(model)
model = AttenFusionModel(img_model, embed_size=embed) # 256+95

In [26]:
model.load_state_dict(torch.load(best_model, weights_only=True))
model = model.to(device)

/home/fangyijie/miniconda3/envs/sam_env/lib/python3.11/site-packages/torch/_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


### Test

Following codes are not needed if k-fold validation is enabled.

In [27]:
repeat_test = 1
criterion = nn.MSELoss() # MSE loss

In [28]:
test_size = len(test_dataloader)
model.eval()
rmse_list = []

for i in range(repeat_test):
    test_loss = 0
    with torch.no_grad():
        for x1, x2, y in test_dataloader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            if IS_ATTENTION:
                pred, att_wgh = model(x1, x2)
            else:
                pred = model(x1, x2)
            loss = torch.sqrt(criterion(pred, y)).item()
            # loss = criterion(pred, y).item()
            test_loss += loss
    print(f"Test loss: {test_loss/test_size:>8f}")
    rmse_list.append(test_loss/test_size)

print(f"Average Test RMSE: {np.mean(rmse_list)}")

Test loss: 11.771187
Average Test RMSE: 11.771186960240206


In [29]:
l1_loss = nn.L1Loss()

In [30]:
test_dataloader = DataLoader(test_data, batch_size=1, shuffle=True)

In [31]:
test_size = len(test_dataloader)
model.eval()
mae_list = []
pred_gt_diff = []
gt = []
all_att_weights = []

for i in range(repeat_test):
    test_loss = 0
    with torch.no_grad():
        for x1, x2, y in test_dataloader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            if IS_ATTENTION:
                pred, att_wgh = model(x1, x2)
                gt.append(y.item())
                all_att_weights.append(att_wgh.cpu().numpy())
            else:
                pred = model(x1, x2)
            loss = l1_loss(pred, y).item()
            test_loss += loss
            pred_gt_diff.append(loss)
    print(f"Test loss: {test_loss/test_size:>8f}")
    mae_list.append(test_loss/test_size)

print(f"Average Test MAE: {np.mean(mae_list)}")
print(f"STD Test MAE: {np.std(mae_list)}")

Test loss: 8.116948
Average Test MAE: 8.116948269677225
STD Test MAE: 0.0


In [69]:
len(gt)

766

In [86]:
def return_topk_features(att_wgh, K=5):
    att_wgh_tensor = torch.from_numpy(att_wgh)
    aggregated_weights = att_wgh_tensor[0].sum(dim=1)
    # Get top-K indices
    topk_indices = torch.topk(aggregated_weights, K, largest=True).indices
    topk_weights = aggregated_weights[topk_indices]
    return topk_indices.cpu().numpy(), topk_weights.cpu().numpy()

- 1st_trimester: weeks 1 - 12  (days 7 - 91)
- 2nd_trimester: weeks 13 - 27 (days 92 - 189)
- 3rd_trimester: weeks 28 -    (days 190 - )

In [87]:
one_trimester = []
two_trimester = []
three_trimester = []

ind = 0
for att_wgh in all_att_weights:
    # print("Attention weights shape:", att_wgh.shape)
    topk_indices, topk_weights = return_topk_features(att_wgh, K=5)
    if gt[ind] <= 91:
        one_trimester.append((topk_indices))
    elif gt[ind] > 91 and gt[ind] <= 189:
        two_trimester.append((topk_indices))
    else:
        three_trimester.append((topk_indices))
    # print("Top-K feature indices:", topk_indices)
    # print("Top-K feature weights:", topk_weights)
    ind += 1
    # break

In [88]:
print(len(one_trimester))
print(len(two_trimester))
print(len(three_trimester))

45
392
329


In [89]:
one_matrix = np.array(one_trimester).reshape(45, 5)  # 2 rows, 3 cols
two_matrix = np.array(two_trimester).reshape(392, 5)  # 2 rows, 3 cols
three_matrix = np.array(three_trimester).reshape(329, 5)  # 2 rows, 3 cols

1st Trimester

In [90]:
unique, counts = np.unique(one_matrix, return_counts=True)
freq = dict(zip(unique, counts))
# Sort by frequency (high → low)
sorted_freq = dict(sorted(freq.items(), key=lambda x: x[1], reverse=True))
print(sorted_freq)

{83: 9, 10: 7, 38: 7, 87: 7, 36: 6, 56: 5, 74: 5, 82: 5, 8: 4, 9: 4, 14: 4, 16: 4, 17: 4, 37: 4, 49: 4, 52: 4, 54: 4, 72: 4, 75: 4, 84: 4, 15: 3, 19: 3, 20: 3, 23: 3, 26: 3, 27: 3, 29: 3, 31: 3, 33: 3, 34: 3, 43: 3, 46: 3, 57: 3, 59: 3, 70: 3, 73: 3, 80: 3, 81: 3, 0: 2, 7: 2, 12: 2, 21: 2, 25: 2, 30: 2, 32: 2, 35: 2, 41: 2, 42: 2, 44: 2, 53: 2, 55: 2, 58: 2, 60: 2, 61: 2, 62: 2, 67: 2, 71: 2, 76: 2, 77: 2, 79: 2, 85: 2, 93: 2, 94: 2, 4: 1, 5: 1, 11: 1, 13: 1, 18: 1, 22: 1, 24: 1, 39: 1, 40: 1, 45: 1, 47: 1, 48: 1, 51: 1, 63: 1, 64: 1, 65: 1, 66: 1, 86: 1, 88: 1, 89: 1, 90: 1, 91: 1}


- 83: gldm_DependenceNonUniformityNormalized
- 10: firstorder_90Percentile
- 38: glcm_JointEntropy
- 87: gldm_HighGrayLevelEmphasis
- 36: glcm_DifferenceVariance

2nd Trimester

In [91]:
unique, counts = np.unique(two_matrix, return_counts=True)
freq = dict(zip(unique, counts))
# Sort by frequency (high → low)
sorted_freq = dict(sorted(freq.items(), key=lambda x: x[1], reverse=True))
print(sorted_freq)

{83: 38, 11: 37, 12: 36, 18: 36, 78: 35, 28: 34, 33: 34, 58: 33, 8: 32, 32: 32, 34: 32, 10: 31, 21: 31, 9: 30, 57: 30, 82: 30, 38: 29, 76: 29, 16: 28, 20: 28, 26: 27, 81: 27, 17: 26, 24: 26, 35: 26, 56: 26, 60: 26, 79: 26, 13: 25, 15: 25, 29: 25, 53: 25, 73: 25, 74: 25, 84: 25, 14: 24, 59: 24, 85: 24, 36: 23, 51: 22, 72: 22, 75: 22, 77: 22, 80: 22, 87: 22, 19: 21, 22: 21, 25: 21, 27: 21, 30: 21, 61: 21, 64: 21, 31: 20, 63: 20, 23: 19, 37: 19, 50: 19, 65: 19, 66: 19, 67: 19, 49: 18, 54: 18, 47: 17, 40: 16, 68: 16, 69: 16, 70: 16, 86: 16, 45: 15, 52: 15, 55: 15, 71: 15, 88: 15, 5: 14, 44: 14, 48: 14, 62: 14, 7: 13, 0: 12, 6: 12, 1: 11, 39: 11, 41: 11, 42: 11, 92: 11, 43: 10, 46: 8, 89: 7, 90: 6, 93: 5, 94: 5, 91: 3, 2: 1, 3: 1}


- 83: gldm_DependenceNonUniformityNormalized
- 11: firstorder_Energy
- 12: firstorder_Entropy
- 18: firstorder_Median
- 78: glszm_ZoneEntropy

3rd Trimester

In [92]:
unique, counts = np.unique(three_matrix, return_counts=True)
freq = dict(zip(unique, counts))
# Sort by frequency (high → low)
sorted_freq = dict(sorted(freq.items(), key=lambda x: x[1], reverse=True))
print(sorted_freq)

{85: 32, 13: 30, 16: 30, 82: 30, 73: 29, 10: 28, 56: 28, 79: 28, 19: 27, 84: 27, 12: 26, 33: 26, 72: 26, 76: 26, 77: 26, 81: 26, 21: 25, 28: 25, 14: 24, 18: 24, 32: 24, 36: 24, 83: 24, 8: 23, 11: 23, 15: 23, 27: 23, 58: 23, 60: 23, 35: 22, 55: 22, 25: 21, 29: 21, 37: 21, 9: 20, 20: 20, 31: 20, 34: 20, 52: 20, 54: 20, 75: 20, 80: 20, 26: 19, 59: 19, 88: 19, 49: 18, 51: 18, 62: 18, 65: 18, 71: 18, 17: 17, 30: 17, 47: 17, 57: 17, 63: 17, 74: 17, 24: 16, 41: 16, 78: 16, 23: 15, 70: 15, 22: 14, 53: 14, 68: 14, 7: 13, 38: 13, 67: 13, 39: 12, 43: 12, 69: 12, 46: 11, 48: 11, 64: 11, 66: 11, 86: 11, 87: 11, 94: 11, 0: 10, 45: 10, 91: 10, 50: 9, 61: 9, 44: 8, 1: 7, 42: 7, 89: 7, 5: 6, 40: 6, 90: 5, 3: 3, 6: 3, 93: 3, 2: 1}


- 85: gldm_GrayLevelNonUniformity
- 13: firstorder_InterquartileRange
- 16: firstorder_MeanAbsoluteDeviation
- 82: gldm_DependenceNonUniformity
- 73: glszm_SizeZoneNonUniformity

### Heatmap

In [40]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
def compute_saliency_map(attention_weights, deep_features, radiomic_features):
    """
    Compute saliency map by applying attention weights to deep features.

    Args:
        attention_weights (torch.Tensor): Attention matrix of shape (256, 99).
        deep_features (torch.Tensor): Deep feature map of shape (256, H, W).
        radiomic_features (torch.Tensor): Radiomic features of shape (99,).

    Returns:
        torch.Tensor: Saliency map of shape (H, W).
    """
    # Compute weighted sum of deep features using attention weights
    weighted_features = torch.matmul(attention_weights, radiomic_features)
    weighted_features = weighted_features.view(-1, 1, 1)  # Reshape to (256, 1, 1)

    # Apply weights to the deep feature map
    saliency_map = (deep_features * weighted_features).sum(dim=0)

    return saliency_map


In [ ]:
def visualize_saliency_map(saliency_map, title="Saliency Map"):
    """
    Visualize the saliency map.

    Args:
        saliency_map (torch.Tensor): Saliency map of shape (H, W).
        title (str): Title for the plot.
    """
    saliency_map = saliency_map.detach().cpu().numpy()
    plt.figure(figsize=(8, 6))
    sns.heatmap(saliency_map, cmap="viridis")
    plt.title(title)
    plt.axis("off")
    plt.show()


In [ ]:
def top_k_radiomic_features(attention_weights, K=10):
    """
    Identify and visualize top-K radiomic features based on attention weights.

    Args:
        attention_weights (torch.Tensor): Attention matrix of shape (256, 95).
        K (int): Number of top radiomic features to display.
    """
    # Aggregate attention weights for each radiomic feature
    aggregated_weights = attention_weights.sum(dim=0)

    # Get top-K indices
    topk_indices = torch.topk(aggregated_weights, K, largest=True).indices
    bottomk_indices = torch.topk(aggregated_weights, K, largest=False).indices
    topk_weights = aggregated_weights[topk_indices]
    bottomk_weights = aggregated_weights[bottomk_indices]

    # weights = [topk_weights, bottomk_weights]
    indices = torch.cat((topk_indices, bottomk_indices), 0)
    weights = torch.cat((topk_weights, bottomk_weights), 0)

    # Plot the top-K features
    plt.figure(figsize=(10, 6))
    # plt.bar(range(K*2), weights.cpu().numpy(), tick_label=indices.cpu().numpy())
    plt.bar(range(K), topk_weights.cpu().numpy(), tick_label=topk_indices.cpu().numpy())
    # plt.bar(range(K), bottomk_weights.cpu().numpy(), tick_label=bottomk_indices.cpu().numpy())
    plt.title(f"Top {K} Radiomic Features by Attention Weight")
    plt.xlabel("Radiomic Feature Index")
    plt.ylabel("Aggregated Attention Weight")
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()

In [ ]:
aggregated_weights = att_wgh[0].sum(dim=0)
K = 10

# Get top-K indices
topk_indices = torch.topk(aggregated_weights, K, largest=True).indices
bottomk_indices = torch.topk(aggregated_weights, K, largest=False).indices
topk_weights = aggregated_weights[topk_indices]
bottomk_weights = aggregated_weights[bottomk_indices]

In [ ]:
indices = torch.cat((topk_indices, bottomk_indices), 0)
weights = torch.cat((topk_weights, bottomk_weights), 0)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(range(K*2), weights.cpu().numpy(), tick_label=indices.cpu().numpy())
plt.title(f"Top {K} Radiomic Features by Attention Weight")
plt.xlabel("Radiomic Feature Index")
plt.ylabel("Aggregated Attention Weight")
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
topk_indices = torch.topk(att_wgh[0], 5, largest=True).indices
# bottomk_indices = torch.topk(aggregated_weights, K, largest=False).indices
topk_weights = att_wgh[0][topk_indices]
# bottomk_weights = aggregated_weights[bottomk_indices]

# weights = [topk_weights, bottomk_weights]
# bottomk_weights.shape

In [ ]:
# Visualize Top-K Radiomic Features
top_k_radiomic_features(att_wgh[0], K=5)

In [ ]:
# Random example data
attention_weights = att_wgh[0].permute(1, 0)
deep_features = torch.rand(256, 32, 32)
radiomic_features = x2[0]

# Compute saliency map
saliency_map = compute_saliency_map(attention_weights, deep_features, radiomic_features)

# Visualize saliency map
visualize_saliency_map(saliency_map)

In [ ]:
# avg_att_wgh = torch.mean(att_wgh, 0).cpu().numpy()
avg_att_wgh = att_wgh.sum(dim=0).cpu().numpy()
# avg_att_wgh = att_wgh[1].cpu().numpy()

In [ ]:
att_wgh[0].shape

In [ ]:
plt.imshow(avg_att_wgh, cmap='viridis', interpolation='nearest')
plt.show()

In [ ]:
for i in range(repeat_test):
    test_loss = 0
    with torch.no_grad():
        for x1, x2, y in test_dataloader:
            x1, x2, y = x1.to(device), x2.to(device), y.to(device)
            if IS_ATTENTION:
                pred, att_wgh = model(x1, x2)
            else:
                pred = model(x1, x2)

            print(att_wgh.shape)
            # att_wgh.cpu().numpy()

            # plt.imshow(avg_att_wgh, cmap='viridis', interpolation='nearest')
            # plt.show()

            break